https://docs.langchain.com/oss/python/langgraph/use-functional-api#extended-example-calling-a-simple-graph-from-the-functional-api


# Creating a simple workflow

https://docs.langchain.com/oss/python/langgraph/use-functional-api#creating-a-simple-workflow


In [8]:
import uuid  # 표준 라이브러리 uuid로 thread_id용 고유 식별자 생성
from langgraph.func import entrypoint, task  # Functional API의 두 핵심 데코레이터
from langgraph.checkpoint.memory import InMemorySaver  # 인메모리 체크포인터 (재시작 시 휘발됨, 프로토타입용)

# @task: 워크플로우 안에서 호출되는 하나의 작업 단위
# entrypoint나 다른 task 내부에서만 호출 가능하며, 호출 시 즉시 실행되지 않고 future를 반환함
@task
def is_even(number: int) -> bool:
    # 숫자가 짝수인지 판별만 하는 단순 로직
    return number % 2 == 0

@task
def format_message(is_even: bool) -> str:
    # is_even 결과를 받아 사람이 읽을 메시지로 변환
    return "The number is even." if is_even else "The number is odd."

# 체크포인터: entrypoint 실행 중 생성되는 각 task의 결과를 저장소에 기록
# InMemorySaver는 프로세스가 살아있는 동안만 유지되는 메모리 기반 저장소
checkpointer = InMemorySaver()

# @entrypoint: 워크플로우의 시작점이자 전체 실행을 관리하는 함수
# checkpointer를 넘기면 이 함수 호출 자체가 Pregel 그래프로 변환되어 persistence/streaming/HITL 기능을 갖게 됨
@entrypoint(checkpointer=checkpointer)
def workflow(inputs: dict) -> str:
    """Simple workflow to classify a number."""
    # task를 호출하면 바로 결과가 나오는 게 아니라 future 객체가 반환됨
    # .result()를 호출해야 실제 실행이 완료되고 값을 받아옴 (동기적으로 대기)
    even = is_even(inputs["number"]).result()
    # 앞 단계 결과(even)를 다음 task의 입력으로 그대로 전달
    return format_message(even).result()

# thread_id: 이 실행을 식별하는 고유 키
# 같은 thread_id로 다시 invoke하면 체크포인트에 이어서 실행(resume)이 가능해짐
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# entrypoint로 정의된 workflow도 일반 그래프처럼 .invoke()로 실행
# inputs 딕셔너리가 workflow 함수의 매개변수로 그대로 전달됨
result = workflow.invoke({"number": 7}, config=config)
print(result)  # "The number is odd." 출력

The number is odd.


# Extended example: Compose an essay with an LLM

In [9]:
import uuid
from langchain.chat_models import init_chat_model  # 모델 provider를 문자열로 지정해 바로 chat model 인스턴스를 만들어주는 헬퍼
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

# init_chat_model: "provider:model명" 또는 모델명만으로도 적절한 provider를 추론해서 ChatModel 객체 생성
# 여기서는 gpt-5-mini를 사용 (OpenAI API 키가 환경변수에 설정되어 있어야 함)
model = init_chat_model('gpt-5-mini')

# @task: LLM 호출처럼 시간이 걸리거나 외부 API를 호출하는 작업 단위
# task 안에 LLM 호출을 넣으면 체크포인트에 그 결과가 저장되어, 재실행 시 같은 호출을 중복하지 않음
@task
def compose_essay(topic: str) -> str:
    """Generate an essay about the given topic."""
    # model.invoke()는 메시지 리스트를 받아 AIMessage 객체를 반환
    # .content로 실제 텍스트(에세이 본문)만 꺼냄
    return model.invoke([
        {"role": "system", "content": "You are a helpful assistant that writes essays."},
        {"role": "user", "content": f"Write an essay about {topic}."}
    ]).content

# 체크포인터: entrypoint 실행 중 task 결과(여기선 LLM 응답)를 저장
# 이게 있어야 같은 thread_id로 재실행 시 LLM을 다시 호출하지 않고 저장된 결과를 재사용 가능
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(topic: str) -> str:
    """Simple workflow that generates an essay with an LLM."""
    # task 호출 → future 반환 → .result()로 실제 LLM 호출이 끝날 때까지 대기 후 결과(str) 받음
    return compose_essay(topic).result()

# thread_id: 이 실행(대화/세션)을 구분하는 키
# 같은 thread_id로 다시 invoke하면 체크포인트 기록을 이어서 사용 가능 (예: interrupt 후 재시작)
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# entrypoint로 정의된 workflow를 .invoke()로 실행
# topic="the history of flight"가 workflow(topic) 인자로 그대로 전달됨
result = workflow.invoke("the history of flight", config=config)
print(result)  # LLM이 생성한 에세이 전체 텍스트 출력

Humans have dreamed of flight for as long as they have told stories. From mythic figures who donned wings to escape the earth to early philosophers who watched birds and wondered how they stayed aloft, the history of flight is a story of imagination, careful observation, repeated failure, and incremental engineering breakthroughs. Over roughly three centuries this curiosity transformed into practical technologies that reshaped economies, warfare, travel, and our understanding of the planet and beyond.

Early ideas and aerodynamic foundations
The impulse to fly appears in the world’s earliest myths—Daedalus and Icarus in Greek legend, winged Hindu and Buddhist deities, and many other traditions—which show how deeply the idea of flight has been embedded in human thought. Practical approaches, however, required scientific insight. In the late 18th and early 19th centuries, English engineer Sir George Cayley laid the groundwork for modern aeronautics. Cayley identified the four forces acti

# Extended example: parallel LLM calls

In [10]:
import uuid
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

# 모델 초기화
model = init_chat_model("gpt-5-mini")

# @task: 주어진 topic 하나에 대해 LLM으로 문단을 생성
@task
def generate_paragraph(topic: str) -> str:
    response = model.invoke([
        {"role": "system", "content": "You are a helpful assistant that writes educational paragraphs."},
        {"role": "user", "content": f"Write a paragraph about {topic}."}
    ])
    return response.content  # AIMessage에서 텍스트만 추출

# 체크포인터: 각 task(LLM 호출) 결과를 저장 → 재실행 시 중복 호출 방지
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(topics: list[str]) -> str:
    """Generates multiple paragraphs in parallel and combines them."""
    # 핵심: task를 호출하는 순간 결과를 기다리지 않고 future를 즉시 반환함
    # 따라서 이 리스트 컴프리헨션은 모든 generate_paragraph 호출을 "동시에" 시작시킴 (병렬 실행)
    futures = [generate_paragraph(topic) for topic in topics]

    # .result()를 호출하는 시점에서야 각 future의 완료를 기다림
    # 이미 위에서 전부 병렬로 시작됐기 때문에, 여기서는 가장 늦게 끝나는 호출 시간만큼만 대기
    paragraphs = [f.result() for f in futures]

    # 모든 문단을 빈 줄 두 개로 구분해서 하나의 문자열로 합침
    return "\n\n".join(paragraphs)

# 워크플로우 실행
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = workflow.invoke(
    ["quantum computing", "climate change", "history of aviation"],
    config=config
)
print(result)  # 세 개의 문단이 순서대로 합쳐진 텍스트 출력

Quantum computing is a new computing paradigm that uses quantum bits, or qubits, which can exist in superpositions of 0 and 1 and become entangled with one another, allowing certain computations to explore many possibilities simultaneously. Operations on qubits are performed with quantum gates that manipulate amplitudes and phases, and cleverly designed algorithms—such as Shor’s algorithm for factoring and Grover’s search algorithm—can solve specific problems much faster than the best-known classical methods. Practical quantum machines are challenged by noise and decoherence, so building large-scale, reliable quantum computers requires error correction and fault-tolerant architectures; current experimental platforms include superconducting circuits, trapped ions, and photonic systems. While quantum computers are unlikely to supplant classical computers for general-purpose tasks, they hold promise for specialized applications in cryptography, chemistry and materials simulation, optimiza